In [19]:
import json
import os 
import pandas as pd
import numpy as np
import pyreadstat

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "kano2018human")
original_data_pathway = os.path.join(pathway, "original_data")

starting_point= original_data_pathway

temp = []

for dirpath, dirnames, filenames in os.walk(starting_point):
    for index, filename in enumerate([f for f in filenames if f.endswith("_edited.csv")]):
        sav_filepath = os.path.join(dirpath, filename)
        # print(sav_filepath)
        x = pd.read_csv(sav_filepath)
        x.columns = map(str.lower, x.columns)
        x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
        x['file_name']=filename
        x['study_id']="kano2018human"
        temp.append(x)
fulldf = pd.concat(temp, ignore_index=True, sort=False)


out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [20]:
fulldf.rename(columns={"group": "group_original",
    "species":"species_original",
    "subject":"ape"}, inplace=True)

fulldf['species_original'].replace('chimp', 'chimpanzee', inplace=True, regex=True)


In [21]:
# fulldf['ape'].unique()

In [22]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')

In [23]:
# fulldf["file_name"] = fulldf["file_name"].astype(str)
fulldf[['experiment_name','experiment', 'temp1', 'temp2']] = fulldf['file_name'].str.split('_',expand=True)

In [24]:
fulldf['experiment'].replace('exp', '', inplace=True, regex=True)
fulldf['experiment'].replace('12', '4', inplace=True, regex=True)

In [25]:
import re
replace_1=re.compile('(\(|\)|\-|\/| |\,)') # / and :
fulldf.columns = fulldf.columns.str.replace(replace_1, '_')
fulldf.columns = fulldf.columns.str.replace('__', '_')
fulldf.replace(' ', '_', inplace=True, regex=True)


In [26]:
spe_2=[]  
for index, row in fulldf.iterrows():
    if not pd.isna(row['species']):
        spe_2.append(row['species'])
    else:
        spe_2.append(row['species_original'])
fulldf = fulldf.assign(species=spe_2)
fulldf.dropna(subset=['ape'], inplace=True)
# fulldf.columns


In [27]:

fulldf.rename(columns={"ape": "participant",
   "order_of_condition_ost_ostension_first_cont_control_first_":"order_of_condition",
    "rearing_mot_mother_nur_nursary_peer_":"rearing"}, inplace=True)



In [28]:
replace_list = [['rearing', 'mot','mother'],
                ['rearing','nur','hand_reared'],
                ['order_of_condition','cont','control'],
                ['order_of_condition','ost','ostension']]
for x,y,z in replace_list:
    fulldf[x].replace(y, z, inplace=True)

In [29]:
# fulldf.columns

In [30]:
fulldf = fulldf[~fulldf.experiment.str.contains("4")]
face_view = fulldf[fulldf.experiment_name.str.contains("face_viewing_time")]
first_look = fulldf[fulldf.experiment_name.str.contains("target_first_look")]
viewing_time = fulldf[fulldf.experiment_name.str.contains("target_viewing_time")]

face_list = face_view[['study_id','experiment', 'participant', 'sex', 'species','group_original','rearing',
       'order_of_condition', 'ostensive_still_phase', 'ostensive_cue_phase',
       'ostensive_look_phase', 'control_still_phase', 'control_cue_phase',
       'control_look_phase']].values.tolist()

first_list = first_look[['experiment', 'participant','ostensive_target',
       'ostensive_distractor', 'control_target', 'control_distractor']].values.tolist()

view_list = viewing_time[['experiment', 'participant','ostensive_target',
       'ostensive_distractor', 'control_target', 'control_distractor']].values.tolist()

In [31]:
combined_lol = [lol_1+lol_2 for lol_1,lol_2 in zip(face_list,first_list)]
combined_lol = [lol_1+lol_2 for lol_1,lol_2 in zip(combined_lol, view_list)]

In [32]:
df = pd.DataFrame(combined_lol, columns=['study_id','experiment', 'participant', 'sex', 'species','group_original','rearing',
       'order_of_condition', 'face_view_time_ostensive_still_phase', 'face_view_time_ostensive_cue_phase',
       'face_view_time_ostensive_look_phase', 'face_view_time_control_still_phase', 'face_view_time_control_cue_phase',
       'face_view_time_control_look_phase', 'experiment_2', 'participant_2','first_look_ostensive_target',
       'first_look_ostensive_distractor', 'first_look_control_target', 'first_look_control_distractor',
       'experiment_3', 'participant_3','viewing_time_ostensive_target',
       'viewing_time_ostensive_distractor', 'viewing_time_control_target', 'viewing_time_control_distractor'])

In [33]:
df['order_of_condition'].replace('eat', 'control_eat', inplace=True, regex=True)

rearing_correction = [
                        # ['kuno','hand_reared'],
                        # ['lexi','hand_reared'],
                        # ['alex','hand_reared'],
                        # ['fraukje','hand_reared'],
                        # ['joey','hand_reared'],
                        # ['riet','hand_reared'],
                        ['robert','hand_reared'],
                        ['natascha','hand_reared'],
                        ['pini','mother'],
                        ['luiza','mother'],
                        ['daza','unknown'],
                        ['frederike','unknown'],
                        ['jeudi','unknown']]
for x,y in rearing_correction:
    df.loc[df.participant == x, ['rearing']] = y


complete_path_non_mpi = os.path.join(original_data_pathway, "non_mpi_participants.csv")
df_non_mpi = pd.read_csv(complete_path_non_mpi)
non_mpi = df_non_mpi.values.tolist()


In [34]:

for x,y in non_mpi:
    df.loc[df.participant == x, ['sex']] = y


df.rename(columns={"group_original": "group_id"}, inplace=True)

In [35]:
fulldf=df[[ 'study_id','experiment', 'participant', 'sex', 'species','group_id',
       'order_of_condition', 'face_view_time_ostensive_still_phase', 'face_view_time_ostensive_cue_phase',
       'face_view_time_ostensive_look_phase', 'face_view_time_control_still_phase', 'face_view_time_control_cue_phase',
       'face_view_time_control_look_phase', 'first_look_ostensive_target',
       'first_look_ostensive_distractor', 'first_look_control_target', 'first_look_control_distractor',
       'viewing_time_ostensive_target',
       'viewing_time_ostensive_distractor', 'viewing_time_control_target', 'viewing_time_control_distractor']]


In [36]:

for index in range(1,4):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'kano2018human_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'kano2018human_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
